# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ma5029blp-wq/ML-flyrank-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub

In [2]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

Paste your Hugging Face READ token (hf_...): ··········


In [3]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
features = con.sql(f"""
WITH bounds AS (
    SELECT MAX(report_date) AS end_d
    FROM {TABLES['fact_daily']}
),
windowed AS (
    SELECT
        f.client_hash_id,
        f.content_hash_id,

        SUM(CASE
            WHEN f.report_date > b.end_d - INTERVAL 30 DAY
            THEN f.gsc_impressions ELSE 0 END) AS imp_last30,

        SUM(CASE
            WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
            THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,

        SUM(CASE
            WHEN f.report_date > b.end_d - INTERVAL 30 DAY
            THEN f.gsc_clicks ELSE 0 END) AS clk_last30,

        AVG(CASE
            WHEN f.report_date > b.end_d - INTERVAL 30 DAY
            THEN f.gsc_avg_position END) AS pos_last30

    FROM {TABLES['fact_daily']} f, bounds b

    WHERE f.report_date > b.end_d - INTERVAL 60 DAY

    GROUP BY 1,2

    HAVING imp_prev30 >= 100
)

SELECT *
FROM windowed
""").df()

print(features.shape)
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(111247, 6)


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_62f4a7e64f5e0096,content_0dac238195631de4,219.0,251.0,0.0,3.468159
1,client_62f4a7e64f5e0096,content_f6116743b00afc2d,995.0,4786.0,2.0,25.024091
2,client_62f4a7e64f5e0096,content_332f337f995e9781,103.0,177.0,0.0,18.600206
3,client_62f4a7e64f5e0096,content_82053c8b9d8a4811,736.0,1518.0,2.0,9.526655
4,client_62f4a7e64f5e0096,content_742939be32c206d2,269.0,335.0,3.0,7.904483


In [5]:
qsignals = con.sql(f"""
SELECT
    content_hash_id,
    ANY_VALUE(content_visible_query_count) AS visible_queries,
    ANY_VALUE(rare_impressions_share) AS rare_share,
    ANY_VALUE(anonymized_impressions_share) AS anon_share,
    MAX(impressions_90d) AS top_query_impressions,
    SUM(impressions_90d) AS kept_impressions

FROM {TABLES['fact_query_90d']}

GROUP BY content_hash_id
""").df()

qsignals["top_query_share"] = (
    qsignals["top_query_impressions"] /
    qsignals["kept_impressions"]
)

qsignals.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,content_447894f2faf0d2bc,14,0.043656,0.725102,55,339.0,0.162242
1,content_4486e5efcc7b773f,10,0.041108,0.748594,169,486.0,0.347737
2,content_448b01e1de5750b4,3,0.137037,0.685185,23,48.0,0.479167
3,content_44a3108ea1a95d57,5,0.075630,0.043697,235,524.0,0.448473
4,content_44c082bdb9a864ea,23,0.004720,0.810897,542,3242.0,0.167181


In [17]:
con.sql(f"""
SELECT content_hash_id, COUNT(DISTINCT client_hash_id) AS n_clients
FROM {TABLES['fact_query_90d']}
GROUP BY content_hash_id
HAVING n_clients > 1
""").df()

,content_hash_id,n_clients


In [6]:
data = features.merge(
    qsignals,
    on="content_hash_id",
    how="left"
)

data["is_declining"] = (
    data["imp_last30"] < 0.8 * data["imp_prev30"]
).astype(int)

print(data.shape)
data.head()

(111247, 13)


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share,is_declining
0,client_62f4a7e64f5e0096,content_0dac238195631de4,219.0,251.0,0.0,3.468159,15.0,0.144623,0.665019,79.0,308.0,0.256494,0
1,client_62f4a7e64f5e0096,content_f6116743b00afc2d,995.0,4786.0,2.0,25.024091,101.0,0.037423,0.178737,15557.0,18432.0,0.844021,1
2,client_62f4a7e64f5e0096,content_332f337f995e9781,103.0,177.0,0.0,18.600206,3.0,0.215054,0.623656,25.0,60.0,0.416667,1
3,client_62f4a7e64f5e0096,content_82053c8b9d8a4811,736.0,1518.0,2.0,9.526655,16.0,0.032740,0.717915,473.0,952.0,0.496849,1
4,client_62f4a7e64f5e0096,content_742939be32c206d2,269.0,335.0,3.0,7.904483,8.0,0.224066,0.630705,30.0,140.0,0.214286,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

The feature vector contains historical performance and query-level features that are available before making a content refresh decision.

Note: only content with imp_prev30 >= 100 is included in this feature set, filtering out very low-traffic content to avoid unstable percentage-decline calculations on near-zero baselines. This means the model only speaks to already-visible content, not new or low-traffic pages.

Verified that content_hash_id maps to exactly one client_hash_id in fact_query_90d (a check for COUNT(DISTINCT client_hash_id) > 1 per content_hash_id returned zero rows), so merging on content_hash_id alone does not risk mixing data across clients.

- imp_prev30 – Total Google Search impressions during the previous 30-day window. Missing values are not expected because impressions are aggregated from the daily table. This feature is available before the prediction.

- visible_queries – Number of visible search queries for the content. Missing values occur when a page has no query-level data, so those rows are removed before modeling. This feature is available before the prediction.

- rare_share – Share of impressions from rare queries. Missing values are handled by removing rows with missing query features. This feature is available before the prediction.

- anon_share – Share of anonymized query impressions. Missing values are handled by removing rows with missing query features. This feature is available before the prediction.

- top_query_share – Percentage of impressions coming from the top-performing query. It is calculated from the query table and is available before the prediction.

- clk_last30, pos_last30 – Computed but not used as model features. Both are calculated from the same last-30-day window used to define is_declining, so including them would leak label information the same way imp_last30 does.

In [13]:
feature_cols = [
    "imp_prev30",
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share"
]

missing = data[feature_cols].isnull().mean().to_frame("missing_rate")
missing

,missing_rate
imp_prev30,0.000000
visible_queries,0.081297
rare_share,0.081297
anon_share,0.081297
top_query_share,0.081297


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

To check for leakage, I first trained a model using only features that were available before the prediction window. This model achieved an accuracy of 65.3%. I then deliberately added the label-derived feature imp_last30, which is used to create the target variable (is_declining). The model accuracy increased to 98.9%, showing that the model was using information from the target itself rather than learning general patterns. This demonstrates label leakage, so imp_last30 was removed from the final feature set because it would not be available at prediction time.

Additional note: the query-level features (visible_queries, rare_share, anon_share, top_query_share) are computed from a 90-day window ending on the same date as imp_last30/imp_prev30. That 90-day window overlaps the last-30-day period the label is built from, so these "honest" features are not perfectly pre-decision — they carry a diluted version of the same leakage as imp_last30, just spread across a longer window. A stricter setup would compute them from a 90-day window ending before the last-30-day period begins.

In [14]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Honest features
honest_features = [
    "imp_prev30",
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share"
]

model_data = data.dropna(subset=honest_features)

X = model_data[honest_features]
y = model_data["is_declining"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

honest_accuracy = accuracy_score(
    y_test,
    model.predict(X_test)
)

print("Honest accuracy:", honest_accuracy)

Honest accuracy: 0.6524206488982819


In [15]:
# Add a leaky feature
leaky_features = [
    "imp_prev30",
    "imp_last30",      # <-- Leaky feature
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share"
]

model_data = data.dropna(subset=leaky_features)

X = model_data[leaky_features]
y = model_data["is_declining"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

leaky_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

leaky_model.fit(X_train, y_train)

leaky_accuracy = accuracy_score(
    y_test,
    leaky_model.predict(X_test)
)

print("Leaky accuracy:", leaky_accuracy)

Leaky accuracy: 0.9892763492622598


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

The following fields were excluded from the final feature set:

- imp_last30 – Excluded because it is used to create the target variable (is_declining), so using it would cause label leakage.

- clk_last30, pos_last30 – Excluded from modeling, since both come from the same last-30-day window used to build is_declining and would leak label information.

- client_hash_id – Used only for identifying clients or grouped validation, not as a model feature.

- content_hash_id – A unique identifier for each content item. It contains no predictive information and should not be used as a feature.

- top_query_impressions – Excluded because top_query_share already captures the query concentration in a normalized form.

- kept_impressions – Excluded because it is only used to calculate top_query_share and would be redundant in the model.

In [16]:
final_features = [
    "imp_prev30",
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share"
]

print("Final features used:")
print(final_features)

print("\nExcluded fields:")
excluded = [
    "imp_last30",
    "clk_last30",
    "pos_last30",
    "client_hash_id",
    "content_hash_id",
    "top_query_impressions",
    "kept_impressions"
]

print(excluded)

Final features used:
['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']

Excluded fields:
['imp_last30', 'clk_last30', 'pos_last30', 'client_hash_id', 'content_hash_id', 'top_query_impressions', 'kept_impressions']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.